# 4.6 Lab: Inference-Time Compute - Token Budget vs Quality Tradeoff[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.6_inference_time_compute/lab.ipynb)[![Open In Molab](https://molab.marimo.io/badge.svg)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.6_inference_time_compute/lab.ipynb)This lab explores the relationship between inference-time compute budget and answer quality through analytical modeling. We simulate scaling laws, budget allocation strategies, and the cascading complexity pattern.

In [ ]:
# Install dependenciesimport subprocesssubprocess.run(["pip", "install", "-q", "numpy", "matplotlib"], check=True)

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# === PARAMETERS (modify and re-run) ===# Test-time scaling law parameters (from Snell et al. 2024 empirical fits)BASELINE_ACCURACY = 0.45  # accuracy with zero reasoning tokensSCALING_COEFFICIENT = 0.12  # accuracy gain per doubling of tokensMAX_ACCURACY = 0.95  # asymptotic ceilingTOKEN_BUDGETS = [0, 64, 256, 1024, 4096, 16384, 65536]  # reasoning token budgets to evaluate

## Experiment 1: Test-Time Scaling LawModel the log-linear relationship between reasoning tokens and accuracy on hard problems (Snell et al. 2024). The scaling law follows:$$\text{accuracy}(t) = \min\left(A_{max},\; A_0 + k \cdot \log_2(t + 1)\right)$$where $t$ is reasoning tokens, $A_0$ is baseline accuracy, and $k$ is the scaling coefficient.

In [ ]:
def accuracy_from_tokens(tokens, baseline=BASELINE_ACCURACY, k=SCALING_COEFFICIENT, cap=MAX_ACCURACY):    """Compute accuracy given reasoning token budget using log-linear scaling law."""    # Log-linear scaling: each doubling of tokens gives k accuracy improvement    acc = baseline + k * np.log2(tokens + 1)    # Cap at maximum achievable accuracy    return np.minimum(acc, cap)# Generate smooth curve across token rangetoken_range = np.logspace(0, 5, 200)  # 1 to 100K tokensaccuracy_curve = accuracy_from_tokens(token_range)# Plot the scaling lawfig, ax = plt.subplots(1, 1, figsize=(10, 5))ax.semilogx(token_range, accuracy_curve * 100, 'b-', linewidth=2)# Mark specific budget tiersfor budget in TOKEN_BUDGETS[1:]:    acc = accuracy_from_tokens(budget)    ax.axvline(budget, color='gray', linestyle='--', alpha=0.3)    ax.plot(budget, acc * 100, 'ro', markersize=8)    ax.annotate(f'{acc*100:.1f}%', (budget, acc*100+1.5), ha='center', fontsize=8)ax.set_xlabel('Reasoning Tokens (log scale)')ax.set_ylabel('Accuracy (%)')ax.set_title('Test-Time Scaling Law: Accuracy vs Reasoning Token Budget')ax.set_ylim(40, 100)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('scaling_law.png', dpi=150, bbox_inches='tight')plt.show()print(f"At 4096 tokens: {accuracy_from_tokens(4096)*100:.1f}% accuracy")print(f"At 65536 tokens: {accuracy_from_tokens(65536)*100:.1f}% accuracy")

## Experiment 2: Cost-Accuracy FrontierCompare strategies: how much does each additional unit of accuracy cost across different inference-time compute approaches?

In [ ]:
# === PARAMETERS ===COST_PER_TOKEN = 0.00001  # $/token for generationVERIFIER_COST_PER_CANDIDATE = 0.0005  # $/candidate for best-of-N verificationN_VALUES = [1, 2, 4, 8, 16, 32, 64]  # best-of-N candidate countsCOT_LENGTHS = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768]  # chain-of-thought budgets

In [ ]:
def best_of_n_accuracy(n, per_candidate_acc=0.55):    """Accuracy of best-of-N: P(at least 1 correct in N tries)."""    # Probability none are correct = (1-p)^N    return 1.0 - (1.0 - per_candidate_acc) ** ndef best_of_n_cost(n, avg_tokens_per_candidate=2048):    """Total cost for best-of-N sampling."""    # N generations + N verifier calls    gen_cost = n * avg_tokens_per_candidate * COST_PER_TOKEN    verify_cost = n * VERIFIER_COST_PER_CANDIDATE    return gen_cost + verify_cost# Chain-of-thought: accuracy scales with token budgetcot_accuracies = [accuracy_from_tokens(t) for t in COT_LENGTHS]cot_costs = [t * COST_PER_TOKEN for t in COT_LENGTHS]# Best-of-N: accuracy scales with candidate countbon_accuracies = [best_of_n_accuracy(n) for n in N_VALUES]bon_costs = [best_of_n_cost(n) for n in N_VALUES]# Plot cost-accuracy frontierfig_4, ax_4 = plt.subplots(1, 1, figsize=(10, 5))ax_4.plot([c*1000 for c in cot_costs], [a*100 for a in cot_accuracies],        'b-o', linewidth=2, markersize=6, label='Chain-of-Thought')ax_4.plot([c*1000 for c in bon_costs], [a*100 for a in bon_accuracies],        'r-s', linewidth=2, markersize=6, label='Best-of-N')ax_4.set_xlabel('Cost per Query (millicents)')ax_4.set_ylabel('Accuracy (%)')ax_4.set_title('Cost-Accuracy Frontier: CoT vs Best-of-N')ax_4.legend()ax_4.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('cost_accuracy_frontier.png', dpi=150, bbox_inches='tight')plt.show()

## Experiment 3: Cascading Complexity SavingsModel the cascading pattern where queries escalate through tiers only when cheaper tiers fail. Compute average cost vs uniform deep reasoning.

In [ ]:
# === PARAMETERS ===# Difficulty distribution of incoming queriesDIFFICULTY_DIST = {    'trivial': 0.30,   # 30% of queries need no reasoning    'easy': 0.35,      # 35% solved with light reasoning    'medium': 0.20,    # 20% need moderate reasoning    'hard': 0.12,      # 12% need deep reasoning    'extreme': 0.03    # 3% need exhaustive search}# Cost multiplier for each tier (relative to base cost of 1 token generation)TIER_COSTS = {    'trivial': 200,     # ~200 tokens, no reasoning    'easy': 2256,       # 256 reasoning + 200 output + overhead    'medium': 4248,     # 2048 reasoning + 200 + overhead    'hard': 34200,      # 32K reasoning + overhead    'extreme': 134400   # best-of-8 x 4K + search}# Uniform deep reasoning cost (applied to ALL queries)UNIFORM_DEEP_COST = 34200

In [ ]:
# Compute average cost with cascading vs uniformcascade_avg_cost = sum(    DIFFICULTY_DIST[tier] * TIER_COSTS[tier]    for tier in DIFFICULTY_DIST)uniform_cost = UNIFORM_DEEP_COST  # every query gets deep reasoningsavings_pct = (1 - cascade_avg_cost / uniform_cost) * 100# Visualize cost distributiontiers = list(DIFFICULTY_DIST.keys())fractions = [DIFFICULTY_DIST[t] for t in tiers]costs = [TIER_COSTS[t] for t in tiers]weighted = [DIFFICULTY_DIST[t] * TIER_COSTS[t] for t in tiers]fig, axes = plt.subplots(1, 2, figsize=(12, 5))# Left: query difficulty distributioncolors = ['#dcfce7', '#dbeafe', '#fef3c7', '#ffedd5', '#ffe4e6']axes[0].bar(tiers, [f*100 for f in fractions], color=colors, edgecolor='black')axes[0].set_ylabel('% of Queries')axes[0].set_title('Query Difficulty Distribution')axes[0].set_ylim(0, 45)# Right: cost comparisonaxes[1].bar(['Cascade\n(adaptive)', 'Uniform\n(deep reasoning)'],            [cascade_avg_cost, uniform_cost],            color=['#dcfce7', '#ffe4e6'], edgecolor='black')axes[1].set_ylabel('Avg Tokens per Query')axes[1].set_title(f'Average Cost: Cascade saves {savings_pct:.0f}%')axes[1].axhline(cascade_avg_cost, color='green', linestyle='--', alpha=0.5)plt.tight_layout()plt.savefig('cascade_savings.png', dpi=150, bbox_inches='tight')plt.show()print(f"Cascade average cost: {cascade_avg_cost:.0f} tokens/query")print(f"Uniform deep cost: {uniform_cost:.0f} tokens/query")print(f"Savings: {savings_pct:.1f}%")

## Experiment 4: KV Cache Pressure Under Reasoning WorkloadsModel how reasoning models reduce concurrent request capacity due to KV cache growth.

In [ ]:
# === PARAMETERS ===GPU_HBM_GB = 80           # A100 80GBMODEL_WEIGHTS_GB = 35     # INT8 70B modelKV_AVAILABLE_GB = GPU_HBM_GB - MODEL_WEIGHTS_GB  # remaining for KV cache# Model config (Llama 70B style)NUM_LAYERS = 80NUM_KV_HEADS = 8HEAD_DIM = 128BYTES_PER_PARAM = 2  # FP16 KV cache# Scenarios: traditional vs reasoningSCENARIOS = {    'Traditional (8K ctx)': 8192,    'Light reasoning (16K)': 16384,    'Medium reasoning (32K)': 32768,    'Deep reasoning (64K)': 65536,    'Exhaustive (128K)': 131072}

In [ ]:
def kv_cache_bytes(seq_len):    """Compute KV cache size in bytes for one request."""    # K and V each: layers * kv_heads * head_dim * seq_len * bytes    per_kv = NUM_LAYERS * NUM_KV_HEADS * HEAD_DIM * seq_len * BYTES_PER_PARAM    return per_kv * 2  # K + V# Compute concurrent capacity for each scenarioscenario_names = list(SCENARIOS.keys())seq_lens = list(SCENARIOS.values())kv_sizes_gb = [kv_cache_bytes(s) / 1e9 for s in seq_lens]max_concurrent = [int(KV_AVAILABLE_GB / kv) for kv in kv_sizes_gb]# Plotfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))# Left: KV cache per requestax1.barh(scenario_names, kv_sizes_gb, color='#dbeafe', edgecolor='black')ax1.set_xlabel('KV Cache per Request (GB)')ax1.set_title('KV Cache Growth with Reasoning Length')ax1.axvline(KV_AVAILABLE_GB, color='red', linestyle='--', label=f'Available: {KV_AVAILABLE_GB}GB')ax1.legend()# Right: concurrent requestsax2.barh(scenario_names, max_concurrent, color='#dcfce7', edgecolor='black')ax2.set_xlabel('Max Concurrent Requests')ax2.set_title(f'Concurrency on {GPU_HBM_GB}GB GPU ({MODEL_WEIGHTS_GB}GB weights)')for i, v in enumerate(max_concurrent):    ax2.text(v + 0.3, i, str(v), va='center', fontsize=10)plt.tight_layout()plt.savefig('kv_cache_pressure.png', dpi=150, bbox_inches='tight')plt.show()for name, kv, conc in zip(scenario_names, kv_sizes_gb, max_concurrent):    print(f"{name:30s} | KV: {kv:.2f} GB | Max concurrent: {conc}")

## Key Takeaways1. **Test-time scaling is real**: accuracy follows a log-linear relationship with reasoning tokens, but with diminishing returns2. **Strategy matters**: Best-of-N is cheaper for verifiable problems; CoT is cheaper for open-ended reasoning3. **Cascading saves 70-85%**: routing easy queries to cheap tiers drastically cuts average cost4. **KV cache is the bottleneck**: reasoning models reduce concurrency 5-15x per GPU, driving the need for disaggregated serving and KV offloading